[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/shells/shells.ipynb)

# Discrete Shells

This example simulates the buckling and twisting of thin elastic shells modelled as triangulated surfaces in three-dimensional space, following Grinspun, Hirani, Desbrun and Schröder, "Discrete shells" (2003). Each edge acts as an axial spring resisting stretching along the surface, while each interior hinge between adjacent triangles resists bending away from its rest dihedral angle. When rest lengths or natural curvatures cannot fit flat in three-dimensional space, a growing disc buckles into a saddle and a bilayer strip with crossed natural curvatures twists into a ribbon.

In three-dimensional projective geometric algebra, joining two vertex points forms their edge line `tails & heads`, and joining an interior hinge line with the opposite corner in each adjacent triangle forms the two intrinsic face patches `line & across` and `-(line & opposite)`. Multiplying the normalized patches and taking the square root `(second.normalized() * first.normalized()).square_root()` gives the motor that folds one face onto the other, whose logarithm paired with the unit hinge line reads half the chiral dihedral angle, while topological coboundary and reading adjugates such as `topology.d[0].adjugate() * pull` assemble edge tensions and hinge reactions onto vertices with zero net force and moment.

In vector-calculus discrete-shell notation, the dihedral angle at interior edge $e = (x_0, x_1)$ between unit face normals $\mathbf{n}_1 = (x_1 - x_0)\times(x_2 - x_0)/\|(x_1 - x_0)\times(x_2 - x_0)\|$ and $\mathbf{n}_2 = (x_0 - x_1)\times(x_3 - x_1)/\|(x_0 - x_1)\times(x_3 - x_1)\|$ requires the two-argument quadrant check $\theta_e = \operatorname{atan2}((\mathbf{n}_1\times\mathbf{n}_2)\cdot\hat{\mathbf{e}},\,\mathbf{n}_1\cdot\mathbf{n}_2)$, and its four vertex gradients split into opposite-corner terms $\nabla_{x_2}\theta_e = \mathbf{n}_1/h_1$, $\nabla_{x_3}\theta_e = \mathbf{n}_2/h_2$ and two cotangent-weighted endpoint combinations along $\hat{\mathbf{e}}$.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import jax
import numpy as np

from examples.mechanics.shells import render, scenarios
from examples.util.animation import save_figure, show_states
from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from numga.backend.jax import JaxContext
from numga.sparse import Sparse
from numga.topology import Topology

np.set_printoptions(precision=3, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                      # Scalar+
Chiral = Scalar.derive.flipped_orientation                                       # Chiral-
Plane = ga.gatype.vector()                                                       # Plane+
Patch = ga.gatype.antitrivector()                                                # Patch-
Point = ga.gatype.antivector()                                                   # Point-
# Lines joining points: edges, hinges, and the action lines of vertex forces.
Line = ga.gatype.antibivector()                                                  # Line-
Motor = ga.gatype.rotor()                                                        # Motor+
# Symmetric form evaluating normal curvature across unit lines in the surface.
Curvature = ga.gatype((Scalar, Line, Line))                                      # Scalar+ <- (Line-, Line-)
frame_duration_ms = 40

## Elastic energy and internal forces

The shell stores stretching energy on every edge from changes in edge length and bending energy on every interior hinge from deviations of the dihedral angle away from its rest angle. Differentiating that energy pulls each edge's endpoints along their shared line and twists each hinge's two incident triangles equally and oppositely: each triangle pushes its opposite corner perpendicular to its patch along `(patch | corner) ^ w`, scaled inversely by the corner's height above the hinge, while the hinge endpoints share the equal and opposite reaction by the lever rule. Because every edge tension and hinge torque is internal, joining each vertex point with its force `settled & forces` sums across the surface to zero net action line.

In [ ]:
def shell_energy(
    topology: Topology,
    hinge_ends: Sparse,
    hinge_corners: Sparse,
    lengths: Scalar,
    angles: Chiral,
    stretching: float,
    bending: Scalar,
    points: Point,
) -> Scalar:
    """Edge stretching against `lengths` (`Scalar+[E]`) and interior-hinge bending against `angles`
    (`Chiral-[E_interior]`) sum to the total elastic energy (`[] Scalar+`) at `points`
    (`Point-[V]`)."""
    tails, heads = topology.incidence[1, 0] * points                             # Point-[E], Point-[E]
    stretched = (tails & heads).norm() - lengths                                 # Scalar+[E]
    ends, tips = hinge_ends * points                                             # Point-[E_interior], Point-[E_interior]
    across, opposite = hinge_corners * points                                    # Point-[E_interior], Point-[E_interior]
    line = ends & tips                                                           # Line-[E_interior]
    # Wind both face patches consistently so a coplanar hinge has unit motor and zero folding angle.
    first, second = line & across, -(line & opposite)                            # Patch-[E_interior], Patch-[E_interior]
    # Multiplying the two unit patches turns twice about their shared hinge line; twice the square root's log reads the angle.
    motor = (second.normalized() * first.normalized()).square_root()             # Motor+[E_interior]
    bent = -2 * (motor.log() | line.normalized()) - angles                       # Chiral-[E_interior]
    stretch_energy = (stretching * stretched * stretched / lengths).sites.sum()  # [] Scalar+
    bend_energy = (bending * bent.squared()).sites.sum()                         # [] Scalar+
    return (stretch_energy + bend_energy) / 2


def shell_forces(
    topology: Topology,
    hinge_ends: Sparse,
    hinge_corners: Sparse,
    lengths: Scalar,
    angles: Chiral,
    stretching: float,
    bending: Scalar,
    points: Point,
) -> Point:
    """Negative energy gradients on edges and interior hinges scatter back through the topological
    adjugates to give the net internal force (`Point-[V]`) at `points` (`Point-[V]`)."""
    tails, heads = topology.incidence[1, 0] * points                             # Point-[E], Point-[E]
    current_lengths = (tails & heads).norm()                                     # Scalar+[E]
    pull = (heads - tails) * (stretching * (current_lengths - lengths) / (lengths * current_lengths))   # Point-[E]

    ends, tips = hinge_ends * points                                             # Point-[E_interior], Point-[E_interior]
    across, opposite = hinge_corners * points                                    # Point-[E_interior], Point-[E_interior]
    line = ends & tips                                                           # Line-[E_interior]
    first, second = line & across, -(line & opposite)                            # Patch-[E_interior], Patch-[E_interior]
    motor = (second.normalized() * first.normalized()).square_root()             # Motor+[E_interior]
    moment = line.norm() * bending * (-2 * (motor.log() | line.normalized()) - angles)   # Chiral-[E_interior]
    # Each face pushes its opposite corner normal to its patch along `(patch | corner) ^ w`, scaled by inverse height.
    pushes = [((patch | corner) ^ mv.w) * (moment / patch.norm_squared())
              for patch, corner in ((first, across), (second, opposite))]        # [2] Point-[E_interior]
    # Divide the equal-and-opposite reaction between the hinge endpoints by where each opposite corner projects along the hinge.
    shares = [((ends & corner) | line) / (line | line) for corner in (across, opposite)]   # [2] Scalar+[E_interior]
    at_tips = -(pushes[0] * shares[0] + pushes[1] * shares[1])                   # Point-[E_interior]
    at_ends = -(pushes[0] + pushes[1]) - at_tips                                 # Point-[E_interior]

    # Pull back edge tensions and hinge corner reactions onto vertices via the coboundary and reading adjugates.
    return (-(topology.d[0].adjugate() * pull)
            + (hinge_ends.adjugate() * stack([at_ends, at_tips])).sum(axis=0)
            + (hinge_corners.adjugate() * stack(pushes)).sum(axis=0))

## Rest geometry and damped relaxation

Interior edges and their opposite corners come directly from `topology.interior[1]` and `topology.facet_opposites`. Evaluating a symmetric curvature form `curvature(unit, unit)` on each hinge's unit line gives the normal curvature across that hinge; multiplying by the centroid-to-centroid span across the two incident triangles converts curvature into the rest dihedral angle, signed toward the chosen top plane via the face patches. Starting from rest with a small out-of-plane perturbation, damped symplectic-Euler steps dissipate kinetic energy until the shell settles into equilibrium.

In [ ]:
def rest_geometry(
    topology: Topology, points: Point, curvature: Curvature, top: Plane, thickness: float,
) -> tuple[Sparse, Sparse, Chiral, Scalar, Scalar]:
    """From a rest configuration (`Point-[V]`) of uniform `thickness`, unit modulus, and unit
    density, `curvature` toward `top` determines the interior-hinge endpoint and opposite-corner
    selectors (`[2] Scalar+[E_interior <- V]`), chiral rest dihedral angles
    (`Chiral-[E_interior]`), hinge bending stiffnesses (`Scalar+[E_interior]`), and lumped vertex
    masses (`Scalar+[V]`)."""
    # Select each interior hinge's two endpoints in the +1 face's winding order and the opposite corners in both faces.
    hinge_ends = topology.reading(topology.interior[1], 1) * topology.incidence[1, 0]   # [2] Scalar+[E_interior <- V]
    hinge_corners = topology.reading(topology.facet_opposites.T, 0)              # [2] Scalar+[E_interior <- V]
    ends, tips = hinge_ends * points                                             # Point-[E_interior], Point-[E_interior]
    across, opposite = hinge_corners * points                                    # Point-[E_interior], Point-[E_interior]
    line = ends & tips                                                           # Line-[E_interior]
    first, second = line & across, -(line & opposite)                            # Patch-[E_interior], Patch-[E_interior]
    # Normal curvature across the hinge times the centroid-to-centroid span (one third of the two heights) sets the rest bend.
    unit = line.normalized()                                                     # Line-[E_interior]
    spans = (first.norm() + second.norm()) / (3 * line.norm())                   # Scalar+[E_interior]
    face_corners = topology.incidence[2, 0] * points                             # [3] Point-[F]
    patches = face_corners[0] & face_corners[1] & face_corners[2]                # Patch-[F]
    # Pairing the face patches with `top` supplies the chiral orientation sign toward the chosen top side.
    facing = (patches | top).sites.sum()                                         # [] Chiral-
    angles = curvature(unit, unit) * spans * (facing / facing.norm())            # Chiral-[E_interior]
    bending = thickness ** 3 / 2 * -(line | line) / (first.norm() + second.norm())   # Scalar+[E_interior]
    # Lumping one third of each triangle's mass onto each of its three corners gives the vertex masses.
    masses = topology.incidence[2, 0].sum(axis=0).adjugate() * patches.norm() * (thickness / 6)   # Scalar+[V]
    return hinge_ends, hinge_corners, angles, bending, masses


def relax(
    topology: Topology,
    hinge_ends: Sparse,
    hinge_corners: Sparse,
    lengths: Scalar,
    angles: Chiral,
    stretching: float,
    bending: Scalar,
    masses: Scalar,
    points: Point,
    step: float,
    damping: float,
    n_frames: int,
    n_steps_per_frame: int,
) -> Iterator[Point]:
    """Yields `n_frames + 1` vertex configurations (`Point-[V]`) spaced `n_steps_per_frame` damped
    symplectic-Euler steps apart as the shell relaxes from rest at `points`."""
    def substep(state: tuple[Point, Point], _) -> tuple[tuple[Point, Point], None]:
        current_points, velocity = state
        acceleration = shell_forces(
            topology, hinge_ends, hinge_corners, lengths, angles, stretching, bending, current_points,
        ) / masses                                                               # Point-[V]
        next_velocity = (velocity + acceleration * step) * (1 - damping)         # Point-[V]
        return (current_points + next_velocity * step, next_velocity), None

    # Compile one frame's worth of damped time steps into a single scanned loop.
    advance = jax.jit(lambda state: jax.lax.scan(substep, state, length=n_steps_per_frame)[0])
    context = JaxContext(ga)
    state = (points.materialize(context), (points - points).materialize(context))   # Point-[V], Point-[V]
    for _ in range(n_frames):
        yield state[0].materialize(mv.context)
        state = advance(state)
    yield state[0].materialize(mv.context)

## Radially growing disc

Scaling rest edge lengths quadratically with radial distance up to 30 percent at the rim gives the perimeter more length than a flat circle of that radius can hold, so the zero-rest-curvature disc buckles out of the plane into a saddle. Running the two thousand damped relaxation steps takes about fifteen seconds.

In [ ]:
disc_radius, disc_spacing, rim_growth, disc_thickness, disc_step = 1.0, 0.1, 0.3, 0.03, 0.02
damping, n_frames, n_steps_per_frame = 0.01, 100, 20
disc_topology, disc_points, disc_perturbed = scenarios.disc_domain(disc_spacing, disc_radius)

In [ ]:
tails, heads = disc_topology.incidence[1, 0] * disc_points                       # Point-[E], Point-[E]
# Scale rest edge lengths quadratically with midpoint distance from the origin.
distance = (mv.w.dual() & ((tails + heads) / 2)).norm() / disc_radius            # Scalar+[E]
disc_lengths = (tails & heads).norm() * (1 + rim_growth * distance * distance)   # Scalar+[E]
zero_curvature = 0 * (Line | Line)                                               # [] Scalar+ <- (Line-, Line-)
disc_ends, disc_corners, disc_angles, disc_bending, disc_masses = rest_geometry(
    disc_topology, disc_points, zero_curvature, mv.z, disc_thickness,
)
disc_frames = show_states(
    relax(
        disc_topology, disc_ends, disc_corners, disc_lengths, disc_angles,
        disc_thickness, disc_bending, disc_masses, disc_perturbed,
        disc_step, damping, n_frames, n_steps_per_frame,
    ),
    render.animate, disc_topology.elements[2],
    name="shells_disc", duration_ms=frame_duration_ms, n_frames=n_frames + 1,
)

## Crossed-curvature seed-pod strip

Two fibrous layers oriented at plus and minus 45 degrees from the strip's centreline bend toward opposite sides of the sheet, forming a hyperbolic natural curvature tensor whose Gauss curvature is incompatible with an unstretched flat metric. Compromising between stretching and bending twists the strip into a helical ribbon. Running the two thousand damped relaxation steps takes about a dozen seconds.

In [ ]:
strip_length, strip_width, strip_spacing, strip_thickness, strip_step = 2.0, 0.3, 0.08, 0.05, 0.012
natural_curvature, pod_angle = 6.0, np.pi / 4
strip_topology, strip_points, strip_perturbed = scenarios.strip_domain(
    strip_spacing, strip_length, strip_width,
)

In [ ]:
tails, heads = strip_topology.incidence[1, 0] * strip_points                     # Point-[E], Point-[E]
strip_lengths = (tails & heads).norm()                                           # Scalar+[E]
# Rotate the strip's centreline either way in the xy-plane to align the two fibrous layers.
axis = mv.wx.dual()                                                              # [] Line-
first_turn = (mv.xy * (-pod_angle / 2)).exp()                                    # [] Motor+
second_turn = (mv.xy * (pod_angle / 2)).exp()                                    # [] Motor+
first_fibre, second_fibre = first_turn >> axis, second_turn >> axis              # [] Line-, [] Line-
curvature = natural_curvature * (
    (second_fibre | Line) * (second_fibre | Line) - (first_fibre | Line) * (first_fibre | Line)
)                                                                                # [] Scalar+ <- (Line-, Line-)
strip_ends, strip_corners, strip_angles, strip_bending, strip_masses = rest_geometry(
    strip_topology, strip_points, curvature, mv.z, strip_thickness,
)
strip_frames = show_states(
    relax(
        strip_topology, strip_ends, strip_corners, strip_lengths, strip_angles,
        strip_thickness, strip_bending, strip_masses, strip_perturbed,
        strip_step, damping, n_frames, n_steps_per_frame,
    ),
    render.animate, strip_topology.elements[2],
    name="shells_strip", duration_ms=frame_duration_ms, n_frames=n_frames + 1,
)

In [ ]:
save_figure(
    render.draw_settled([
        (disc_topology.elements[2], disc_frames[-1]),
        (strip_topology.elements[2], strip_frames[-1]),
    ]),
    "shells_settled",
);

Both settled surfaces appear side by side, lit according to the orientation of each triangle's patch: the radially grown disc settles into a saddle on the left, while the crossed-curvature strip settles into a twisted ribbon on the right.

In [ ]:
# checks
for topology, hinge_ends, hinge_corners, lengths, angles, stretching, bending, frames, min_span in (
    (disc_topology, disc_ends, disc_corners, disc_lengths, disc_angles, disc_thickness, disc_bending, disc_frames, 0.3),
    (strip_topology, strip_ends, strip_corners, strip_lengths, strip_angles, strip_thickness, strip_bending, strip_frames, 0.1),
):
    initial, settled = frames[0], frames[-1]
    assert shell_energy(
        topology, hinge_ends, hinge_corners, lengths, angles, stretching, bending, settled,
    ) < shell_energy(
        topology, hinge_ends, hinge_corners, lengths, angles, stretching, bending, initial,
    )
    # Internal forces produce zero net force and zero net moment (vanishing total action line).
    total = (settled & shell_forces(
        topology, hinge_ends, hinge_corners, lengths, angles, stretching, bending, settled,
    )).sites.sum()                                                               # [] Line-
    assert total.scalar_norm_squared() + total.dual().scalar_norm_squared() < 1e-24
    heights = (mv.z & settled).batch()                                           # [V] Scalar+
    assert heights.max() - heights.min() > min_span
tails, heads = disc_topology.incidence[1, 0] * disc_frames[-1]                   # Point-[E], Point-[E]
strain = ((tails & heads).norm() / disc_lengths - 1).batch().abs()               # [E] Scalar+
assert strain.mean() < 0.02